In [5]:
import os

print(os.getcwd())
print(os.listdir())
import os

image_dir = "healthy_images"

files = os.listdir(image_dir)

print(len(files))
print(files[:5])

/Users/konishimiharu/zemi/select
['test_images', 'valid.csv', 'healthy_images', '.DS_Store', 'val_images', 'healthy.ipynb', 'train_1.csv', 'complicated.ipynb', 'test.csv', 'healthy_images.csv', 'train_images']
1434
['4a5a6efc0bef.png', '94076a9fb9b5.png', 'b09101adb478.png', 'd74ccc796517.png', 'a821b6ecef33.png']


In [ ]:
import os
import random
import shutil

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm


# ==========================
# 設定
# ==========================

skeleton_dir = "vessel_output/skeletons"
original_image_dir = "healthy_images"

output_csv = "vessel_complexity_results.csv"

top10_output_dir = "complexity_top10"
debug_output_dir = "complexity_debug"

top_k = 10
debug_sample_count = 5
random_seed = 42


# ==========================
# ファイル名変換
# ==========================

def get_image_id(skeleton_file):

    if skeleton_file.endswith("_vessel_skeleton.png"):
        return skeleton_file.replace(
            "_vessel_skeleton.png",
            ""
        )

    return os.path.splitext(
        skeleton_file
    )[0]


# ==========================
# Branch Point
# ==========================

def count_branch_points(skeleton):

    skeleton_bool = skeleton > 0

    count = 0

    ys, xs = np.where(skeleton_bool)

    for y, x in zip(ys, xs):

        y0 = max(0, y - 1)
        y1 = min(
            skeleton.shape[0],
            y + 2
        )

        x0 = max(0, x - 1)
        x1 = min(
            skeleton.shape[1],
            x + 2
        )

        neighbors = np.sum(
            skeleton_bool[y0:y1, x0:x1]
        ) - 1

        if neighbors >= 3:
            count += 1

    return count


# ==========================
# End Point
# ==========================

def count_endpoints(skeleton):

    skeleton_bool = skeleton > 0

    count = 0

    ys, xs = np.where(skeleton_bool)

    for y, x in zip(ys, xs):

        y0 = max(0, y - 1)
        y1 = min(
            skeleton.shape[0],
            y + 2
        )

        x0 = max(0, x - 1)
        x1 = min(
            skeleton.shape[1],
            x + 2
        )

        neighbors = np.sum(
            skeleton_bool[y0:y1, x0:x1]
        ) - 1

        if neighbors == 1:
            count += 1

    return count


# ==========================
# Fractal Dimension
# ==========================

def fractal_dimension(binary_image):

    Z = binary_image > 0

    if np.sum(Z) < 10:
        return 0.0, None, None

    def boxcount(Z, k):

        S = np.add.reduceat(
            np.add.reduceat(
                Z,
                np.arange(
                    0,
                    Z.shape[0],
                    k
                ),
                axis=0
            ),
            np.arange(
                0,
                Z.shape[1],
                k
            ),
            axis=1
        )

        return np.count_nonzero(S)

    p = min(Z.shape)

    n = 2 ** int(
        np.floor(np.log2(p))
    )

    Z = Z[:n, :n]

    sizes = 2 ** np.arange(
        int(np.log2(n)),
        1,
        -1
    )

    counts = []

    for size in sizes:

        counts.append(
            boxcount(
                Z,
                size
            )
        )

    sizes = np.array(sizes)
    counts = np.array(counts)

    valid = counts > 0

    sizes = sizes[valid]
    counts = counts[valid]

    if len(counts) < 2:
        return 0.0, sizes, counts

    coeffs = np.polyfit(
        np.log(sizes),
        np.log(counts),
        1
    )

    fd = -coeffs[0]

    return fd, sizes, counts


# ==========================
# Fractal Plot
# ==========================

def save_fractal_plot(
    sizes,
    counts,
    fd,
    save_path
):

    if sizes is None:
        return

    plt.figure(
        figsize=(6, 4)
    )

    plt.plot(
        np.log(sizes),
        np.log(counts),
        "o-"
    )

    plt.xlabel("log(Box Size)")
    plt.ylabel("log(Box Count)")
    plt.title(
        f"FD = {fd:.4f}"
    )

    plt.tight_layout()

    plt.savefig(save_path)
    plt.close()


# ==========================
# Box可視化
# ==========================

def save_boxcount_visualization(
    skeleton,
    image_id,
    save_dir
):

    os.makedirs(
        save_dir,
        exist_ok=True
    )

    binary = skeleton > 0

    p = min(binary.shape)

    n = 2 ** int(
        np.floor(np.log2(p))
    )

    binary = binary[:n, :n]

    box_sizes = [
        128,
        64,
        32,
        16
    ]

    for box_size in box_sizes:

        if box_size >= n:
            continue

        vis = cv2.cvtColor(
            (binary.astype(np.uint8) * 255),
            cv2.COLOR_GRAY2BGR
        )

        count = 0

        for y in range(
            0,
            n,
            box_size
        ):
            for x in range(
                0,
                n,
                box_size
            ):

                region = binary[
                    y:y+box_size,
                    x:x+box_size
                ]

                if np.any(region):

                    count += 1

                    cv2.rectangle(
                        vis,
                        (x, y),
                        (
                            x + box_size,
                            y + box_size
                        ),
                        (0, 0, 255),
                        1
                    )

        cv2.imwrite(
            os.path.join(
                save_dir,
                f"{image_id}_box_{box_size}_count_{count}.png"
            ),
            vis
        )


# ==========================
# 正規化
# ==========================

def normalize_column(col):

    mn = col.min()
    mx = col.max()

    if mx <= mn:
        return np.zeros(
            len(col)
        )

    return (
        col - mn
    ) / (
        mx - mn
    )


# ==========================
# Main
# ==========================

os.makedirs(
    top10_output_dir,
    exist_ok=True
)

os.makedirs(
    debug_output_dir,
    exist_ok=True
)

files = sorted([
    f
    for f in os.listdir(
        skeleton_dir
    )
    if f.endswith(".png")
])

random.seed(
    random_seed
)

debug_files = set(
    random.sample(
        files,
        min(
            debug_sample_count,
            len(files)
        )
    )
)

results = []

debug_index = 1

for file in tqdm(files):

    skeleton_path = os.path.join(
        skeleton_dir,
        file
    )

    vessel_skeleton = cv2.imread(
        skeleton_path,
        cv2.IMREAD_GRAYSCALE
    )

    if vessel_skeleton is None:
        continue

    image_id = get_image_id(file)

    fd, sizes, counts = fractal_dimension(
        vessel_skeleton
    )

    branch_points = count_branch_points(
        vessel_skeleton
    )

    end_points = count_endpoints(
        vessel_skeleton
    )

    skeleton_length = int(
        np.sum(
            vessel_skeleton > 0
        )
    )

    results.append({
        "image_id": image_id,
        "fractal_dimension": fd,
        "branch_points": branch_points,
        "end_points": end_points,
        "skeleton_length": skeleton_length
    })

    if file in debug_files:

        sample_dir = os.path.join(
            debug_output_dir,
            f"sample{debug_index:02d}_{image_id}"
        )

        os.makedirs(
            sample_dir,
            exist_ok=True
        )

        save_boxcount_visualization(
            vessel_skeleton,
            image_id,
            os.path.join(
                sample_dir,
                "boxcount"
            )
        )

        save_fractal_plot(
            sizes,
            counts,
            fd,
            os.path.join(
                sample_dir,
                "fractal_plot.png"
            )
        )

        debug_index += 1


df = pd.DataFrame(results)

df["fd_norm"] = normalize_column(
    df["fractal_dimension"]
)

df["branch_norm"] = normalize_column(
    df["branch_points"]
)

df["length_norm"] = normalize_column(
    df["skeleton_length"]
)

df["complexity_score"] = (
    ##仮→今の所branchなしがいいかも
      0.70 * df["fd_norm"]
    + 0.30 * df["branch_norm"]
)


df = df.sort_values(
    "complexity_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = np.arange(
    1,
    len(df) + 1
)

df.to_csv(
    output_csv,
    index=False
)

print(df.head(top_k))


top_df = df.head(top_k)

for _, row in top_df.iterrows():

    image_id = row["image_id"]

    src = os.path.join(
        original_image_dir,
        image_id + ".png"
    )

    dst = os.path.join(
        top10_output_dir,
        f"rank{int(row['rank']):02d}_{image_id}.png"
    )

    if os.path.exists(src):
        shutil.copy2(
            src,
            dst
        )

print()
print("保存完了")
print(output_csv)
print(top10_output_dir)
print(debug_output_dir)

100%|██████████| 1434/1434 [04:20<00:00,  5.51it/s]

       image_id  fractal_dimension  branch_points  end_points  \
0  1d2472849dce           1.641682           5105         849   
1  9f8112c710be           1.631245           5276         906   
2  8fbb2ca39911           1.630036           4759         798   
3  7f6690fa390a           1.596851           4795         272   
4  a6731dd737af           1.633571           3933         921   
5  757e39293591           1.597230           3894         642   
6  a7b03e58a6e1           1.607813           3647         710   
7  b4b04d81acbb           1.616400           3313         753   
8  ce754234d760           1.609330           3401         780   
9  29192375ab1b           1.607079           3376         832   

   skeleton_length   fd_norm  branch_norm  length_norm  complexity_score  rank  
0            27686  1.000000     0.966937     0.392558          0.990081     1  
1            26710  0.981219     1.000000     0.377519          0.986854     2  
2            26155  0.979046     0.900039

In [54]:
import os

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw, ImageFont


# ==========================
# 設定
# ==========================

# FD単独の結果CSV
result_csv = "fractal_dimension_results.csv"

# 全体スコア、分岐数の結果CSV
overall_score_csv = "vessel_complexity_results.csv"

# 眼底画像ディレクトリ
image_dir = "healthy_images"

# スケルトン画像ディレクトリ
skeleton_image_dir = "vessel_output/skeletons"

# 出力先
graph_dir = "complexity_graphs"

top_k = 10

os.makedirs(
    graph_dir,
    exist_ok=True
)


# ==========================
# 数値表示用
# ==========================

def format_metric(value):
    if isinstance(value, int) or isinstance(value, np.integer):
        return str(int(value))

    if isinstance(value, float) or isinstance(value, np.floating):
        if float(value).is_integer():
            return str(int(value))

        return f"{value:.4f}"

    return str(value)


# ==========================
# 通常画像読み込み関数
# ==========================

def load_rgb_image(path, size=(256, 256)):
    img = cv2.imread(path)

    if img is None:
        return None

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    img = cv2.resize(
        img,
        size,
        interpolation=cv2.INTER_AREA
    )

    return img


# ==========================
# 画像パス取得
# ==========================

def get_image_path_from_name(image_dir, image_name):
    """
    image_name が xxx.png の場合はそのまま使う。
    image_name が xxx の場合は xxx.png, xxx.jpg, xxx.jpeg,
    xxx.tif, xxx.tiff の順に探す。
    """

    image_name = str(image_name)

    path = os.path.join(
        image_dir,
        image_name
    )

    if os.path.exists(path):
        return path

    root, ext = os.path.splitext(image_name)

    if ext == "":
        candidate_exts = [
            ".png",
            ".jpg",
            ".jpeg",
            ".tif",
            ".tiff"
        ]

        for candidate_ext in candidate_exts:
            candidate_path = os.path.join(
                image_dir,
                image_name + candidate_ext
            )

            if os.path.exists(candidate_path):
                return candidate_path

    return path


# ==========================
# matplotlib版 contact sheet
# 眼底画像など通常画像用
# ==========================

def save_contact_sheet(
    top_df,
    image_column,
    score_column,
    image_dir,
    save_path,
    title,
    score_label,
    extra_columns=None
):
    cols = 5
    rows = int(
        np.ceil(
            len(top_df) / cols
        )
    )

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(18, 8),
        constrained_layout=False
    )

    axes = np.array(
        axes
    ).flatten()

    for i, (_, row) in enumerate(top_df.iterrows()):
        image_name = row[image_column]

        path = get_image_path_from_name(
            image_dir,
            image_name
        )

        img = load_rgb_image(path)

        ax = axes[i]

        if img is not None:
            ax.imshow(img)
        else:
            ax.text(
                0.5,
                0.5,
                "Image not found",
                ha="center",
                va="center",
                fontsize=10
            )

        ax.axis("off")

        title_text = (
            f"Rank {int(row['rank'])}\n"
            f"{score_label} = {format_metric(row[score_column])}"
        )

        if extra_columns is not None:
            for col, label in extra_columns:
                if col in row.index:
                    title_text += (
                        f"\n{label} = {format_metric(row[col])}"
                    )

        ax.set_title(
            title_text,
            fontsize=10,
            pad=8
        )

    for j in range(len(top_df), len(axes)):
        axes[j].axis("off")

    fig.suptitle(
        title,
        fontsize=16,
        y=0.98
    )

    plt.subplots_adjust(
        left=0.03,
        right=0.97,
        top=0.86,
        bottom=0.05,
        wspace=0.08,
        hspace=0.45
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ==========================
# スケルトン画像をそのまま貼る contact sheet
# リサイズなし
# matplotlibなし
# ==========================

def save_skeleton_contact_sheet_original_size(
    top_df,
    image_column,
    score_column,
    image_dir,
    save_path,
    title,
    score_label,
    extra_columns=None,
    cols=5,
    title_height=90,
    margin=30,
    cell_padding=20
):
    """
    スケルトン画像を元解像度のまま貼り付ける。
    cv2.resize や matplotlib の imshow を使わない。
    """

    images = []
    info_rows = []

    for _, row in top_df.iterrows():
        image_name = row[image_column]

        path = get_image_path_from_name(
            image_dir,
            image_name
        )

        if not os.path.exists(path):
            images.append(None)
            info_rows.append(row)
            continue

        img = Image.open(path).convert("RGB")

        images.append(img)
        info_rows.append(row)

    valid_images = [
        img
        for img in images
        if img is not None
    ]

    if len(valid_images) == 0:
        raise ValueError(
            "有効なスケルトン画像が見つかりませんでした。"
        )

    max_w = max(
        img.width
        for img in valid_images
    )

    max_h = max(
        img.height
        for img in valid_images
    )

    rows = int(
        np.ceil(
            len(top_df) / cols
        )
    )

    cell_w = max_w + cell_padding * 2
    cell_h = max_h + title_height + cell_padding * 2

    canvas_w = cols * cell_w + margin * 2
    canvas_h = rows * cell_h + margin * 2 + 60

    canvas = Image.new(
        "RGB",
        (canvas_w, canvas_h),
        color=(255, 255, 255)
    )

    draw = ImageDraw.Draw(canvas)

    try:
        font_title = ImageFont.truetype(
            "DejaVuSans.ttf",
            28
        )

        font_text = ImageFont.truetype(
            "DejaVuSans.ttf",
            18
        )

    except:
        font_title = ImageFont.load_default()
        font_text = ImageFont.load_default()

    draw.text(
        (margin, 15),
        title,
        fill=(0, 0, 0),
        font=font_title
    )

    for i, row in enumerate(info_rows):
        r = i // cols
        c = i % cols

        x0 = margin + c * cell_w
        y0 = margin + 60 + r * cell_h

        img = images[i]

        title_text = (
            f"Rank {int(row['rank'])}  "
            f"{score_label} = {format_metric(row[score_column])}"
        )

        if extra_columns is not None:
            for col, label in extra_columns:
                if col in row.index:
                    title_text += (
                        f"  {label} = {format_metric(row[col])}"
                    )

        draw.text(
            (x0 + cell_padding, y0),
            title_text,
            fill=(0, 0, 0),
            font=font_text
        )

        image_y = y0 + title_height

        if img is not None:
            paste_x = x0 + cell_padding + (
                max_w - img.width
            ) // 2

            paste_y = image_y + cell_padding + (
                max_h - img.height
            ) // 2

            canvas.paste(
                img,
                (paste_x, paste_y)
            )

            draw.rectangle(
                [
                    paste_x,
                    paste_y,
                    paste_x + img.width - 1,
                    paste_y + img.height - 1
                ],
                outline=(180, 180, 180),
                width=1
            )

        else:
            draw.text(
                (
                    x0 + cell_padding,
                    image_y + cell_padding
                ),
                "Image not found",
                fill=(255, 0, 0),
                font=font_text
            )

    canvas.save(save_path)


# ==========================
# FD結果CSV読み込み
# ==========================

df = pd.read_csv(result_csv)

required_cols = [
    "image",
    "fractal_dimension"
]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(
            f"{result_csv} に {col} 列がありません。"
        )


# ==========================
# FD単独でランキング
# ==========================

df = df.sort_values(
    "fractal_dimension",
    ascending=False
).reset_index(drop=True)

df["rank"] = np.arange(
    1,
    len(df) + 1
)

top_df = df.head(top_k).copy()

mean_fd = df["fractal_dimension"].mean()
median_fd = df["fractal_dimension"].median()
std_fd = df["fractal_dimension"].std()

top_df["mean_fractal_dimension"] = mean_fd
top_df["median_fractal_dimension"] = median_fd
top_df["std_fractal_dimension"] = std_fd

if std_fd == 0 or np.isnan(std_fd):
    top_df["z_score"] = 0.0
else:
    top_df["z_score"] = (
        top_df["fractal_dimension"] - mean_fd
    ) / std_fd

top_df.to_csv(
    os.path.join(
        graph_dir,
        "complexity_top10_summary.csv"
    ),
    index=False
)


# ==========================
# 1. FD Top10の棒グラフ
# ==========================

plt.figure(figsize=(12, 6))

plt.bar(
    top_df["rank"].astype(str),
    top_df["fractal_dimension"]
)

plt.axhline(
    mean_fd,
    linestyle="--",
    label=f"Mean = {mean_fd:.4f}"
)

plt.axhline(
    median_fd,
    linestyle=":",
    label=f"Median = {median_fd:.4f}"
)

plt.xlabel("Rank")
plt.ylabel("Fractal Dimension")
plt.title("Top 10 Images by Fractal Dimension")
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        graph_dir,
        "01_top10_fractal_dimension_bar.png"
    ),
    dpi=300
)

plt.close()


# ==========================
# 2. FDの全画像ヒストグラム
# ==========================

plt.figure(figsize=(10, 6))

plt.hist(
    df["fractal_dimension"],
    bins=30
)

plt.axvline(
    mean_fd,
    linestyle="--",
    label=f"Mean = {mean_fd:.4f}"
)

plt.axvline(
    median_fd,
    linestyle=":",
    label=f"Median = {median_fd:.4f}"
)

for value in top_df["fractal_dimension"]:
    plt.axvline(
        value,
        alpha=0.25
    )

plt.xlabel("Fractal Dimension")
plt.ylabel("Number of Images")
plt.title("Distribution of Fractal Dimension in Healthy Images")
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        graph_dir,
        "02_fractal_dimension_histogram.png"
    ),
    dpi=300
)

plt.close()


# ==========================
# 3. FDランク曲線
# ==========================

plt.figure(figsize=(12, 6))

plt.plot(
    df["rank"],
    df["fractal_dimension"],
    marker="o",
    markersize=2,
    linewidth=1
)

plt.scatter(
    top_df["rank"],
    top_df["fractal_dimension"],
    s=60,
    label="Top 10"
)

plt.axhline(
    mean_fd,
    linestyle="--",
    label=f"Mean = {mean_fd:.4f}"
)

plt.xlabel("Rank")
plt.ylabel("Fractal Dimension")
plt.title("Fractal Dimension Ranked in Descending Order")
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        graph_dir,
        "03_rank_curve_fractal_dimension.png"
    ),
    dpi=300
)

plt.close()


# ==========================
# 4. FD Top10 眼底画像のサムネイル一覧
# ==========================

save_contact_sheet(
    top_df=top_df,
    image_column="image",
    score_column="fractal_dimension",
    image_dir=image_dir,
    save_path=os.path.join(
        graph_dir,
        "04_top10_image_contact_sheet.png"
    ),
    title="Top 10 Fundus Images by Fractal Dimension",
    score_label="FD"
)


# ==========================
# 5. FD箱ひげ図
# ==========================

plt.figure(figsize=(6, 6))

plt.boxplot(
    df["fractal_dimension"],
    vert=True
)

plt.scatter(
    np.ones(len(top_df)),
    top_df["fractal_dimension"],
    s=50,
    label="Top 10"
)

plt.ylabel("Fractal Dimension")
plt.title("Top 10 Position in Overall Distribution")
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        graph_dir,
        "05_boxplot_top10_position.png"
    ),
    dpi=300
)

plt.close()


# ==========================
# 6. 全体スコアTop10
#    スケルトン画像 + 眼底画像の両方を出力
# ==========================

overall_df = pd.read_csv(overall_score_csv)

required_overall_cols = [
    "image_id",
    "fractal_dimension",
    "branch_points",
    "complexity_score"
]

for col in required_overall_cols:
    if col not in overall_df.columns:
        raise ValueError(
            f"{overall_score_csv} に {col} 列がありません。"
        )

overall_df = overall_df.sort_values(
    "complexity_score",
    ascending=False
).reset_index(drop=True)

overall_df["rank"] = np.arange(
    1,
    len(overall_df) + 1
)

overall_top_df = overall_df.head(top_k).copy()

overall_top_df["image_file"] = (
    overall_top_df["image_id"].astype(str)
    + ".png"
)

overall_top_df["skeleton_file"] = (
    overall_top_df["image_id"].astype(str)
    + "_vessel_skeleton.png"
)

overall_top_df.to_csv(
    os.path.join(
        graph_dir,
        "overall_complexity_top10_summary.csv"
    ),
    index=False
)


# ==========================
# 6-A. 全体スコアTop10 スケルトン画像
# ==========================

save_skeleton_contact_sheet_original_size(
    top_df=overall_top_df,
    image_column="skeleton_file",
    score_column="complexity_score",
    image_dir=skeleton_image_dir,
    save_path=os.path.join(
        graph_dir,
        "06_top10_overall_complexity_skeleton_contact_sheet.png"
    ),
    title="Top 10 Skeleton Images by Overall Complexity Score",
    score_label="Score",
    extra_columns=[
        ("fractal_dimension", "FD"),
        ("branch_points", "Branch")
    ],
    cols=5,
    title_height=90,
    margin=30,
    cell_padding=20
)


# ==========================
# 6-B. 全体スコアTop10 眼底画像
# ==========================

save_contact_sheet(
    top_df=overall_top_df,
    image_column="image_file",
    score_column="complexity_score",
    image_dir=image_dir,
    save_path=os.path.join(
        graph_dir,
        "08_top10_overall_complexity_fundus_contact_sheet.png"
    ),
    title="Top 10 Fundus Images by Overall Complexity Score",
    score_label="Score",
    
)


# ==========================
# 7. 分岐数Top10 眼底画像のサムネイル一覧
#    表記は Branch のみ
# ==========================

branch_df = pd.read_csv(overall_score_csv)

required_branch_cols = [
    "image_id",
    "branch_points"
]

for col in required_branch_cols:
    if col not in branch_df.columns:
        raise ValueError(
            f"{overall_score_csv} に {col} 列がありません。"
        )

branch_df = branch_df.sort_values(
    "branch_points",
    ascending=False
).reset_index(drop=True)

branch_df["rank"] = np.arange(
    1,
    len(branch_df) + 1
)

branch_top_df = branch_df.head(top_k).copy()

branch_top_df["image_file"] = (
    branch_top_df["image_id"].astype(str)
    + ".png"
)

branch_top_df.to_csv(
    os.path.join(
        graph_dir,
        "branch_points_top10_summary.csv"
    ),
    index=False
)

save_contact_sheet(
    top_df=branch_top_df,
    image_column="image_file",
    score_column="branch_points",
    image_dir=image_dir,
    save_path=os.path.join(
        graph_dir,
        "07_top10_branch_points_contact_sheet.png"
    ),
    title="Top 10 Fundus Images by Branch Points",
    score_label="Branch",
    extra_columns=None
)


# ==========================
# 結果表示
# ==========================

print("グラフを保存しました。")
print(f"保存先: {graph_dir}")

print()
print("FD Top10 summary:")
print(
    top_df[
        [
            "rank",
            "image",
            "fractal_dimension",
            "z_score"
        ]
    ]
)

print()
print("Overall Complexity Top10 summary:")
print(
    overall_top_df[
        [
            "rank",
            "image_id",
            "image_file",
            "skeleton_file",
            "complexity_score",
            "fractal_dimension",
            "branch_points"
        ]
    ]
)

print()
print("Branch Points Top10 summary:")
print(
    branch_top_df[
        [
            "rank",
            
        ]
    ]
)

グラフを保存しました。
保存先: complexity_graphs

FD Top10 summary:
   rank             image  fractal_dimension   z_score
0     1  60aa4e649abf.png           1.249573  1.153389
1     2  6107a2e9f60e.png           1.229233  1.070211
2     3  537e5c578f40.png           1.222656  1.043320
3     4  1e742358e0b9.png           1.220907  1.036166
4     5  5a93c0f783c4.png           1.212704  1.002620
5     6  d1cf31577a59.png           1.211861  0.999173
6     7  e30a890600e1.png           1.204960  0.970954
7     8  70d0392397de.png           1.196338  0.935699
8     9  9095d43fb132.png           1.195089  0.930591
9    10  4276b82e4489.png           1.193980  0.926054

Overall Complexity Top10 summary:
   rank      image_id        image_file                     skeleton_file  \
0     1  1d2472849dce  1d2472849dce.png  1d2472849dce_vessel_skeleton.png   
1     2  9f8112c710be  9f8112c710be.png  9f8112c710be_vessel_skeleton.png   
2     3  8fbb2ca39911  8fbb2ca39911.png  8fbb2ca39911_vessel_skeleton.png  